# N=96: the 2×2 over data budget × lr schedule (job 49641051)

**New best at N=96: 0.9453 [0.9326, 0.9557]** on the shared n=1500 pool, up **+0.0180** on cell B's 0.9273 (88–61, paired McNemar exact **p = 0.033**).

**The finding that matters more:** the two levers are **sub-additive**. Cosine lr is worth +0.0473 at 20,000 instances but only +0.0180 at 80,000; 4× data is worth +0.1007 under a constant lr but only +0.0713 under cosine. Stacking both gives +0.1187 where additivity predicts +0.1480 — an interaction of **−0.0293**.

## Provenance

| | |
|---|---|
| Slurm job | `49641051` (`sud2n96stk`), COMPLETED 11:40:56, ExitCode 0:0 |
| sbatch | `scripts/slurm/sudoku2_n96_stack_levers.sbatch` |
| config | N=96, `d_model=256`, `n_heads=4`, `num_layers=12`, `ff=1024`, `lr=1e-3` **cosine**, 150 epochs, `n_train=80000` |
| seed | training seed as in cell B; eval pool `seed_start=5e10`, instance *i* at `seed_start+i` |
| out | `$SCRATCH/spin_gnn/sudoku_n96_stack_2026-10-26/` |
| the other 3 cells | `49524030` (20k constant), `49565433` (cell A = 20k cosine, cell B = 80k constant) |

One change from cell B: the lr schedule. The checkpoint tag collides with cell A's by design (`n_train` is not in the tag), which `sudoku_agg_eval` disambiguates with the `" [run_dir]"` suffix — that is why the CSV rows below carry directory names.

> **These are decoding scores, not deduction scores.** All four cells were trained on the leaked token order (finding 16): `adj[v]` arrives sorted by the neighbour's true colour, and the N=96 record collapses to **0.0375** when `adj[v]` is permuted. Nothing in this notebook is evidence about reasoning.

In [ ]:
import json, os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binomtest

SCRATCH = os.environ["SCRATCH"]
RUN = f"{SCRATCH}/spin_gnn/sudoku_n96_stack_2026-10-26"
pool = pd.read_csv(f"{RUN}/sharedpool_n1500.csv")
pool[["checkpoint", "solve_rate", "wilson_lo", "wilson_hi", "delta",
      "discordant_won", "discordant_lost", "mcnemar_exact_p"]].round(4)

## 1. Metrics table — the 2×2

The CSV scores every checkpoint against cell B as the reference. The remaining pairwise edges are recomputed below from the per-instance outcomes, so each of the four edges of the square is a paired test on the same 1500 instances.

In [ ]:
z = np.load(f"{RUN}/sharedpool_n1500_per_instance.npz", allow_pickle=True)
S, names = z["solved"], [str(x) for x in z["names"]]
idx = {"B (80k const)": 0, "stack (80k cos)": 1, "A (20k cos)": 2, "base (20k const)": 3}
assert S.shape == (4, 1500)

def mcnemar(a, b):
    A, Bv = S[idx[a]], S[idx[b]]
    won, lost = int((A & ~Bv).sum()), int((~A & Bv).sum())
    p = binomtest(won, won + lost, 0.5).pvalue if won + lost else 1.0
    return dict(comparison=f"{a} vs {b}", delta=A.mean() - Bv.mean(),
                won=won, lost=lost, p=p)

edges = pd.DataFrame([
    mcnemar("A (20k cos)", "base (20k const)"),      # cosine at 20k
    mcnemar("stack (80k cos)", "B (80k const)"),     # cosine at 80k
    mcnemar("B (80k const)", "base (20k const)"),    # data at constant
    mcnemar("stack (80k cos)", "A (20k cos)"),       # data at cosine
    mcnemar("stack (80k cos)", "base (20k const)"),  # both
]).set_index("comparison")
edges.round(4)

In [ ]:
grid = pd.DataFrame(
    [[S[idx["base (20k const)"]].mean(), S[idx["B (80k const)"]].mean()],
     [S[idx["A (20k cos)"]].mean(), S[idx["stack (80k cos)"]].mean()]],
    index=["constant", "cosine"], columns=[20000, 80000])
grid.index.name, grid.columns.name = "lr schedule", "n_train"
additive = (grid.loc["cosine", 20000] - grid.loc["constant", 20000]) \
         + (grid.loc["constant", 80000] - grid.loc["constant", 20000])
observed = grid.loc["cosine", 80000] - grid.loc["constant", 20000]
print(grid.round(4))
print(f"\nobserved both-levers gain {observed:+.4f}, additive prediction {additive:+.4f}, "
      f"interaction {observed - additive:+.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
w, x = 0.36, np.arange(2)
for k, sched in enumerate(["constant", "cosine"]):
    ax.bar(x + (k - 0.5) * w, grid.loc[sched].values, w, label=sched)
ax.set_xticks(x); ax.set_xticklabels(["20,000", "80,000"])
ax.set_xlabel("training instances"); ax.set_ylabel("shared-pool solve_rate")
ax.set_ylim(0.75, 1.0); ax.axhline(0.90, color="r", ls=":", lw=1); ax.legend()
ax.set_title("N=96, n=1500 shared pool")
fig.tight_layout()

## 2. val_nll went the wrong way

The new best has a **higher** final `val_nll` than the cell it beats. The two runs differ in exactly one thing — the lr schedule — so this is not a confounded cross-model comparison of the kind finding 4 was built from. It is a within-family counterexample.

In [ ]:
HIST = {
    "base (20k const)": f"{SCRATCH}/spin_gnn/sudoku_n96_baseline_2026-10-25/d0256_h04_L12_ff01024_lr1e-03_pre",
    "A (20k cos)":      f"{SCRATCH}/spin_gnn/sudoku_n96_plateau_2026-10-26/d0256_h04_L12_ff01024_lr1e-03_pre_cos",
    "B (80k const)":    f"{SCRATCH}/spin_gnn/sudoku_n96_plateau_2026-10-26/d0256_h04_L12_ff01024_lr1e-03_pre",
    "stack (80k cos)":  f"{RUN}/d0256_h04_L12_ff01024_lr1e-03_pre_cos",
}
rows = []
for name, p in HIST.items():
    h = json.load(open(f"{p}/history.json"))
    last, v = h[-1], [e["nll"] for e in h]
    i = int(np.argmin(v))
    rows.append(dict(cell=name, epochs=last["epoch"], train_nll=last["train_nll"],
                     val_nll=last["nll"], gap=last["train_nll"] - last["nll"],
                     val_min=v[i], val_min_epoch=h[i]["epoch"],
                     token_acc=last["accuracy"],
                     solve_rate=S[idx[name]].mean()))
nll = pd.DataFrame(rows).set_index("cell")
nll.round(4)

`val_nll` 1.4014 (new best) vs 1.3948 (cell B): **worse by 0.0066**, while solve_rate is **better by +0.0180 at p = 0.033**. Token accuracy is identical to four places (0.5831 both). The train−val gap is +0.0221 against cell B's +0.0239 — unchanged — and per finding 16 that quantity is not a fit diagnostic in the first place (`train_nll` is averaged over the epoch while the weights update).

## 3. Depth bins

1500 fresh instances at each requested density; realized density is identical across bins within a density, so these are depth effects, not density effects.

In [ ]:
d = pd.read_csv(f"{RUN}/depth_n96_stacked.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, rho in zip(axes, [0.75, 0.6]):
    g = d[d.rho_requested == rho]
    ax.plot(g["T"], g.solve_rate, "o-")
    ax.axhline(0.90, color="r", ls=":", lw=1.2)
    ax.set_xlabel("DD reasoning depth T"); ax.set_ylabel("solve_rate")
    ax.set_ylim(0.85, 1.01)
    ax.set_title(f"rho={rho}, realized {g.realized_rho.iloc[0]:.5f}")
fig.tight_layout()
d[["rho_requested", "T", "n", "realized_rho", "solve_rate", "well_formed_rate"]].round(4)

Every rho=0.60 bin clears 0.90 (weakest T=5 at 0.932, n=146), against cell B whose weakest bin was T=9 at 0.907. There is no monotone decline in T — consistent with finding 11 and *inconsistent* with finding 17's leak-free depth signal, which is the expected contrast: under the leak, depth does not bind.

## 4. Pre-registrations, as resolved

| | claim | verdict | evidence |
|---|---|---|---|
| (a) | It beats cell B's 0.9273 paired at p < 0.05 | **CONFIRMED** | +0.0180, 88–61, p = 0.033. New best at N=96. |
| (b) | The gain is **smaller** than cosine's +0.0473 at 20000 (falsifier: a gain ≥ +0.0473) | **CONFIRMED** | +0.0180, i.e. 2.6× smaller. Explicitly flagged as low confidence because it rested on the same train/val-gap reasoning that was wrong by 2.1× in finding 15. This time it held. |
| (c) | Every rho=0.60 bin still clears 0.90 | **CONFIRMED** | 0.988 / 0.932 / 0.972 / 0.971 / 0.971 / 0.977 over T = 4..9. |
| (d) | The train−val gap widens past +0.0239 | **FALSIFIED** | +0.0221, narrower by 0.0018 — unchanged. The prediction should not have been made: finding 16 had already established that this quantity is not a fit diagnostic. |

## 5. Caveats

- **Leaked token order.** Every number here is a decoding score; the N=96 record collapses to 0.0375 under `adj[v]` permutation (finding 16).
- One seed per cell, so the interaction (−0.0293) is a single-draw estimate of a difference of differences.
- The depth bins come from one draw of 1500 instances, n = 43..146 per bin.
- `val_min` is reported but no checkpoint was selected on it — every checkpoint is the final epoch (open problem in STATUS; worth +0.0280 at N=48).

## 6. Next

The N=96 optimisation question is answered well enough. The run worth compute is the same recipe **leak-free**, which at N=24 costs 0.996 → 0.854 and is where reasoning depth finally bites (findings 16–17).